[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Guji-AnaSora/semantica/blob/main/cookbook_zh/introduction/10b_Graph_Metrics.ipynb)

> **社区拓展篇** —— cookbook_zh 自建教程，非上游翻译。以 [10_Graph_Analytics](./10_Graph_Analytics.ipynb) 为母篇，母篇变更时本篇应复查。

# 图度量指标（Graph Metrics）—— 系统全景

## 概览

母篇用 Semantica 内置工具回答了读图四问：指标体检、度中心性、社区、连通。本篇把量尺配齐——系统过一遍图度量指标全家桶，讲清每个指标**量的是什么、直觉是什么、在知识图谱里怎么用**。

内置工具没覆盖的指标（中介、接近、特征向量中心性、PageRank、密度直径、模块度、k-核）用 **networkx** 补齐。它是 Semantica 的核心依赖，无需另装。

**文档**：[kg API 参考](https://semantica.readthedocs.io/reference/kg/) · [networkx 文档](https://networkx.org/documentation/stable/reference/algorithms/centrality.html)

### 先看主线

图度量看着多，其实只回答三类问题。本篇按这三问分三段，外加一个实战收尾：

**备图（第 1 步）—— 一张埋好故事的示例图**

两个社团（苹果系、微软系）、一座桥（Gartner，唯一的跨社团通道）、一个孤岛（Initech）——每个指标冠军都是设计出来的。

**图级体检（第 2 步）—— 这张图健康吗**

密度（热闹程度）、连通分量（断成几块）、直径与平均路径（几跳可达）、聚类系数（朋友圈抱团程度）。

**节点级（第 3–4 步）—— 谁重要**

中心性四兄弟（度、中介、接近、特征向量）+ PageRank。五种指标在示例图上各选各的冠军——这正是重点：**"重要"的定义不同，冠军就不同**。

**群体级（第 5 步）—— 谁一伙**

社区发现（切圈子）、模块度（切得好不好）、k-核（层层剥洋葱剩下的骨干）。

**实战（第 6 步）—— KG 特有的坑**

知识图谱是带类型的异构图：直接拿全图算中心性，结果会被高频关系类型淹没。演示按关系类型过滤后指标排名如何变化。

### 学习目标

- 说出每类中心性指标量的是什么、适合回答什么问题
- 用 networkx 计算图级指标（密度、直径、聚类系数、连通分量）
- 区分特征向量中心性与 PageRank
- 理解模块度与 k-核的用途
- 在异构知识图谱上正确使用图度量（按关系类型过滤）

## 安装

```bash
pip install semantica
# networkx 是核心依赖，随 semantica 一并安装
```

---

## 第 1 步：备图——埋好故事的示例图

示例图的**设计意图**先摆出来，后面每步对照：

| 埋点 | 节点 | 意在演示 |
|---|---|---|
| 社团中心 | Apple Inc.、Microsoft | 度中心性、特征向量中心性冠军 |
| 跨社团桥 | Gartner（`advises` 两家公司） | 中介、接近中心性冠军 |
| 孤岛 | Initech、Partner Co | 连通分量、k-核剥除 |
| 社团内三角 | Tim/iPhone/Cupertino 互连等 | 聚类系数、社区发现 |

In [ ]:
!pip install semantica


## 第 2 步：图级体检——这张图健康吗

五个指标各量一件事：

- **密度（Density）**：实际边数 ÷ 理论最大边数。聚会的"热闹程度"，0 到 1。KG 里太低往往说明关系抽取覆盖不足
- **连通分量（Components）**：图断成几块。理想的知识图谱是一整块；多出的孤岛要么缺桥、要么混入了无关数据源
- **直径 / 平均路径（Diameter / Avg Path）**：最远两点几跳、平均几跳。路径短意味着推理链条紧凑。注意：**只能在连通图上算**，多分量时取最大分量
- **聚类系数（Clustering）**："朋友的朋友也互相是朋友"的概率。KG 里高聚类往往对应一个主题域

In [ ]:
from semantica.kg import GraphBuilder
import networkx as nx

# 示例图谱：两个社团（苹果系 / 微软系）+ 一座桥（Gartner）+ 一个孤岛（Initech）
# 社团内部刻意三角化，让每个指标都能讲出清楚的故事
entities = [
    {"id": "apple",   "type": "Organization", "name": "Apple Inc.",    "properties": {}},
    {"id": "tim",     "type": "Person",       "name": "Tim Cook",      "properties": {}},
    {"id": "cup",     "type": "Location",     "name": "Cupertino",     "properties": {}},
    {"id": "iphone",  "type": "Product",      "name": "iPhone",        "properties": {}},
    {"id": "msft",    "type": "Organization", "name": "Microsoft",     "properties": {}},
    {"id": "satya",   "type": "Person",       "name": "Satya Nadella", "properties": {}},
    {"id": "redmond", "type": "Location",     "name": "Redmond",       "properties": {}},
    {"id": "azure",   "type": "Product",      "name": "Azure",         "properties": {}},
    {"id": "gartner", "type": "Organization", "name": "Gartner",       "properties": {}},
    {"id": "initech", "type": "Organization", "name": "Initech",       "properties": {}},
    {"id": "partner", "type": "Organization", "name": "Partner Co",    "properties": {}},
]

relationships = [
    # 苹果系（三角化：tim-iphone、tim-cup、iphone-cup 互连）
    {"source": "tim",     "target": "apple",   "type": "ceo_of",       "properties": {}},
    {"source": "apple",   "target": "cup",     "type": "located_in",   "properties": {}},
    {"source": "apple",   "target": "iphone",  "type": "develops",     "properties": {}},
    {"source": "tim",     "target": "iphone",  "type": "leads",        "properties": {}},
    {"source": "tim",     "target": "cup",     "type": "lives_in",     "properties": {}},
    {"source": "iphone",  "target": "cup",     "type": "assembled_in", "properties": {}},
    # 微软系（同样三角化）
    {"source": "satya",   "target": "msft",    "type": "ceo_of",       "properties": {}},
    {"source": "msft",    "target": "redmond", "type": "located_in",   "properties": {}},
    {"source": "msft",    "target": "azure",   "type": "develops",     "properties": {}},
    {"source": "satya",   "target": "redmond", "type": "lives_in",     "properties": {}},
    {"source": "azure",   "target": "redmond", "type": "hosted_in",    "properties": {}},
    # 桥：Gartner 是唯一的跨社团通道
    {"source": "gartner", "target": "apple",   "type": "advises",      "properties": {}},
    {"source": "gartner", "target": "msft",    "type": "advises",      "properties": {}},
    # 孤岛：与主分量完全断开
    {"source": "initech", "target": "partner", "type": "partners_with","properties": {}},
]

kg = GraphBuilder().build({"entities": entities, "relationships": relationships})
id2name = {e["id"]: e["name"] for e in kg["entities"]}
print(f"Graph built: {len(kg['entities'])} entities, {len(kg['relationships'])} relationships")

# 转成 networkx：有向图保留关系类型，另备无向视图给多数指标用
G = nx.DiGraph()
for e in kg["entities"]:
    G.add_node(e["id"], name=e["name"], type=e["type"])
for r in kg["relationships"]:
    G.add_edge(r["source"], r["target"], type=r["type"])
UG = G.to_undirected()
print(f"networkx: {UG.number_of_nodes()} nodes, {UG.number_of_edges()} edges (undirected view)")

## 第 3 步：中心性四兄弟——谁重要

四个指标都问"谁重要"，但对**重要**的定义不同：

| 指标 | 一句话直觉 | 找出什么 |
|---|---|---|
| 度中心性 | 好友数 | 交际花 |
| 中介中心性 | 必经之路 | 跨社团联络人、桥 |
| 接近中心性 | 到哪都近 | 消息传得最快的人 |
| 特征向量中心性 | 朋友都重要 | 圈内核心 |

对示例图的预期，回头对照输出：**度** 的冠军是两家社团中心（Apple、Microsoft 并列）；**中介** 和 **接近** 的冠军是 Gartner——社团间唯一的通道，删掉它图就断成两截；**特征向量** 的冠军是 Apple——它的邻居（Tim、Cupertino、iPhone）彼此也互相连接，是个真核心。

In [ ]:
# 图级体检：规模、密度、连通性、路径、聚类
print(f"Nodes: {UG.number_of_nodes()}, Edges: {UG.number_of_edges()}")
print(f"Density: {nx.density(UG):.3f}")  # 实际边数 / 理论最大边数

components = list(nx.connected_components(UG))
print(f"Connected components: {len(components)}")
for i, comp in enumerate(components, 1):
    print(f"  [{i}] {sorted(id2name[n] for n in comp)}")

# 直径 / 平均路径只能在连通图上算——取最大分量
main = UG.subgraph(max(components, key=len))
print(f"\nMain component: diameter={nx.diameter(main)}, "
      f"avg_path={nx.average_shortest_path_length(main):.2f}")

# 聚类系数 + 三角形：朋友圈抱团程度
print(f"Avg clustering: {nx.average_clustering(UG):.3f}, "
      f"triangles: {sum(nx.triangles(UG).values()) // 3}")

## 第 4 步：PageRank——特征向量的随机游走改良版

PageRank 与特征向量中心性同宗，多了两个修正：连接多的节点把权重**摊薄**再分出去；加入**随机跳转**概率，避免悬挂节点吸走全部权重。直觉：网上随机冲浪的人最终停在每个页面的概率。

注意它在**有向图**上算——方向信息进入了排名。对照第 3 步会发现冠军完全不同：地点类节点（Cupertino、Redmond）因为"被多条边指向"而登顶。这不是谁对谁错，而是**无向的'热闹'与有向的'权威'本来就是两回事**——KG 里选哪个，取决于你把关系方向当语义还是当噪声。

In [ ]:
# 中心性四兄弟：同一张图，四个冠军
metrics = {
    "degree":      nx.degree_centrality(UG),                     # 好友数
    "betweenness": nx.betweenness_centrality(UG),                # 必经之路
    "closeness":   nx.closeness_centrality(UG),                  # 到哪都近
    "eigenvector": nx.eigenvector_centrality(UG, max_iter=1000), # 朋友都重要
}
for name, scores in metrics.items():
    top3 = sorted(scores.items(), key=lambda x: -x[1])[:3]
    pretty = ", ".join(f"{id2name[k]}={v:.3f}" for k, v in top3)
    print(f"{name:12s}: {pretty}")

## 第 5 步：社区、模块度、k-核——谁一伙

- **社区发现（Community Detection）**：把图切成内部连接密、对外连接稀的群。networkx 的贪心模块度算法是一路；Semantica 内置的 `CommunityDetector` 是另一路，正好对照
- **模块度（Modularity）**：给一种切法打分——社区内部边比随机期望多多少。切分是结果，模块度是"切得好不好"。经验上 > 0.3 说明社区结构显著
- **k-核（k-core）**：反复剥掉连接数少于 k 的节点，剩下的核。剥洋葱剥到最后，就是网络最紧密的骨干——本图孤岛在 2-核里会被剥掉

In [ ]:
# PageRank：面向有向图的随机游走排名
pr = nx.pagerank(G, alpha=0.85)
for k, v in sorted(pr.items(), key=lambda x: -x[1])[:3]:
    print(f"{id2name[k]:15s} {v:.3f}")

## 第 6 步：实战——异构图的坑与选型

知识图谱是**带类型的图**：`ceo_of`、`located_in`、`develops`、`advises` 混在同一张图里。直接拿全图算中心性，结果常被高频关系类型淹没。

演示：只保留 `located_in` + `advises` 两类边，切到"地理 + 咨询"视角重算度中心性——Gartner 的绝对分数没变（0.2），相对排名却从第一梯队之外跃升到并列第一，社团中心的优势被稀释。**先想清楚问题涉及哪些关系类型，再过滤，再算**。

选型速查：

| 场景 | 用什么 |
|---|---|
| 检索入口、RAG 种子节点 | 高接近 / 高 PageRank 实体 |
| 评估抽取覆盖度 | 密度、平均度 |
| 数据质量排查 | 连通分量（孤岛）、社区（混入主题） |
| 主题域划分、可视化分组 | 社区发现 |
| 图谱导航稳健性 | 中介中心性（桥节点别丢） |

In [ ]:
# 社区 + 模块度：networkx 贪心算法 vs Semantica 内置检测器
comms_nx = nx.community.greedy_modularity_communities(UG)
print("greedy modularity communities:")
for i, c in enumerate(comms_nx, 1):
    print(f"  [{i}] {sorted(id2name[n] for n in c)}")
print(f"modularity: {nx.community.modularity(UG, comms_nx):.3f}")

from semantica.kg import CommunityDetector
result = CommunityDetector().detect_communities(kg)
print(f"Semantica CommunityDetector: {len(result.get('communities', []))} communities")

# k-core：剥掉度 < k 的节点后剩下的核。孤岛在这里被剥掉
core2 = nx.k_core(UG, k=2)
print(f"\n2-core: {sorted(id2name[n] for n in core2)}")

## 小结

### 回顾主线

三类问题各有一套量尺，示例图上各指标冠军汇总：

| 问题 | 指标 | 冠军 | 为什么 |
|---|---|---|---|
| 图健康吗 | 密度 / 分量 / 直径 / 聚类 | — | 两个分量、直径 4、聚类 0.59，孤岛一眼可见 |
| 谁重要 | 度 | Apple / Microsoft | 各自社团的中心 |
| 谁重要 | 中介 / 接近 | Gartner | 社团间唯一通道 |
| 谁重要 | 特征向量 | Apple | 邻居互相连接的真核心 |
| 谁重要（有向） | PageRank | Cupertino / Redmond | 被多条边指向 |
| 谁一伙 | 社区 + 模块度 0.48 | 苹果系 / 微软系 / 孤岛 | 三群各归各位 |

最后一条实战心得带走：**异构图先过滤关系类型，再算指标**——"重要"永远是相对于你问的问题而言的。

### 延伸

- 母篇 [10_Graph_Analytics](./10_Graph_Analytics.ipynb)：Semantica 内置分析工具
- 上游 [advanced/02_Advanced_Graph_Analytics](../../cookbook/advanced/02_Advanced_Graph_Analytics.ipynb)：把分析放进验证→去重→建图→溯源的完整场景
- [16_Visualization](./16_Visualization.ipynb)：把本篇的社区、中心性画出来

In [ ]:
# 异构图的坑：按关系类型过滤后再算指标
# 只保留 located_in / advises 两类边——切到“地理 + 咨询”视角
F = nx.Graph()
F.add_nodes_from(G.nodes)
for u, v, d in G.edges(data=True):
    if d["type"] in ("located_in", "advises"):
        F.add_edge(u, v)

deg_full = nx.degree_centrality(UG)
deg_view = nx.degree_centrality(F)
print(f"{'entity':16s} {'full graph':>11s} {'geo+advises':>12s}")
for node in sorted(UG, key=lambda n: -deg_full[n])[:5]:
    print(f"{id2name[node]:16s} {deg_full[node]:>11.3f} {deg_view.get(node, 0.0):>12.3f}")